# PaddleOCR Benchmark Notebook

In [78]:
import os
import time
import cv2
import json
from pathlib import Path

import paddle
import onnxruntime

import fitz  # pymupdf
from paddleocr import PaddleOCR, TableRecognitionPipelineV2, DocPreprocessor

_THREADS = {"intra_op_num_threads": os.cpu_count(), "inter_op_num_threads": 1}

print("paddle version:", paddle.__version__)
print("paddle compiled with CUDA:", paddle.is_compiled_with_cuda())
print("onnxruntime version:", onnxruntime.__version__)
print("onnxruntime providers:", onnxruntime.get_available_providers())
print("cpu count:", os.cpu_count())

paddle version: 3.2.2
paddle compiled with CUDA: False
onnxruntime version: 1.28.0
onnxruntime providers: ['AzureExecutionProvider', 'CPUExecutionProvider']
cpu count: 12


## Sample Documents

In [79]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

BENCHMARK_DOCS = sorted(Path("documents/invoices").glob("*.pdf"))
assert BENCHMARK_DOCS, "no sample PDFs found under documents/invoices"

INPUT_DOC = BENCHMARK_DOCS[2]
INPUT_DOC, BENCHMARK_DOCS

(PosixPath('documents/invoices/invoice-sample-3.pdf'),
 [PosixPath('documents/invoices/invoice-sample-1.pdf'),
  PosixPath('documents/invoices/invoice-sample-2.pdf'),
  PosixPath('documents/invoices/invoice-sample-3.pdf')])

## Pipeline

### Preprocessing Pipeline

In [80]:
doc_preprocessor = DocPreprocessor(
    use_doc_orientation_classify=True,
    use_doc_unwarping=True,
    engine="onnxruntime",
    cpu_threads=os.cpu_count()
)

Creating model: ('PP-LCNet_x1_0_doc_ori', None, 'onnxruntime')
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/virtuozs/.paddlex/official_models/PP-LCNet_x1_0_doc_ori_onnx`.
2026-09-10 08:57:48.833057911 [W:onnxruntime:, graph.cc:5566 CleanUnusedInitializersAndNodeArgs] Removing initializer 'p2o.pd_op.full.0.0'. It is not used by any node and should be removed from the model.
2026-09-10 08:57:48.833092966 [W:onnxruntime:, graph.cc:5566 CleanUnusedInitializersAndNodeArgs] Removing initializer 'p2o.pd_op.full_int_array.6.0'. It is not used by any node and should be removed from the model.
2026-09-10 08:57:48.833095626 [W:onnxruntime:, graph.cc:5566 CleanUnusedInitializersAndNodeArgs] Removing initializer 'p2o.pd_op.full_int_array.3.0'. It is not used by any node and should be removed from the model.
2026-09-10 08:57:48.833097753 [W:onnxruntime:, graph.cc:5566 CleanUnusedInitializersAndNodeArgs] Removing initializer 'p2o.pd_op.ful

### OCR Pipeline

In [81]:
ocr_pipeline = PaddleOCR(
    text_detection_model_name="PP-OCRv6_medium_det",
    text_recognition_model_name="PP-OCRv6_medium_rec",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    return_word_box=True,
    enable_mkldnn= True,
    use_textline_orientation=False,
    engine="onnxruntime",
    cpu_threads=os.cpu_count()
)

Creating model: ('PP-OCRv6_medium_det', None, 'onnxruntime')
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/virtuozs/.paddlex/official_models/PP-OCRv6_medium_det_onnx`.


Creating model: ('PP-OCRv6_medium_rec', None, 'onnxruntime')
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/virtuozs/.paddlex/official_models/PP-OCRv6_medium_rec_onnx`.


## Runtime

In [82]:
def page_to_markdown(page):
    lines_sorted = sorted(page["lines"], key=lambda lb: (lb[1][1], lb[1][0])) 
    return {"markdown_texts": "\n".join(text for text, _box , _scores, _words, _word_boxes in lines_sorted), "markdown_images": {}}

In [83]:
def save_to_markdown(page, page_path, save_dir):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    md = page_to_markdown(page)
    md_path = save_dir / f"{Path(page_path).stem}.md"
    md_path.write_text(md["markdown_texts"], encoding="utf-8")
    return md_path

In [84]:
def page_to_json(page):
    lines = []
    for idx, (text, box, score, words, word_boxes) in enumerate(page["lines"]):
        lines.append({
            "id": idx,
            "text": text,
            "bbox": [round(float(c), 2) for c in box],
            "confidence": round(float(score), 4),
            "words": [
                {"text": w, "bbox": [round(float(c), 2) for c in wb]}
                for w, wb in zip(words, word_boxes)
            ],
        })
    height, width = page["preprocessed_image"].shape[:2]
    return {"image_width": width, "image_height": height, "lines": lines}

In [85]:
def save_to_json(page, page_path, save_dir):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    data = page_to_json(page)
    json_path = save_dir / f"{Path(page_path).stem}.json"
    json_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
    return json_path

In [86]:
def ingest_document(pdf_path, zoom=2.0):
    pdf_path = Path(pdf_path)
    doc_dir = OUTPUT_DIR / f"{pdf_path.stem}"
    doc_dir.mkdir(parents=True, exist_ok=True)

    doc = fitz.open(pdf_path)
    mat = fitz.Matrix(zoom, zoom)

    pages = []
    for i, page in enumerate(doc):
        page_dir = doc_dir / f"page-{i}"
        page_dir.mkdir(parents=True, exist_ok=True)

        page_path = page_dir / f"page-{i}.png"
        page.get_pixmap(matrix=mat).save(str(page_path))

        t0 = time.perf_counter()
        preprocessed = list(doc_preprocessor.predict(input=str(page_path)))[0]
        t1 = time.perf_counter()
        page_image = preprocessed["output_img"]

        preprocessed_path = page_dir / f"page-{i}-preprocessed.png"
        cv2.imwrite(str(preprocessed_path), page_image)

        ocr_result = list(ocr_pipeline.predict(input=page_image))[0]
        t2 = time.perf_counter()

        page_data = {
            "lines": list(zip(
                ocr_result["rec_texts"],
                ocr_result["rec_boxes"].tolist(),
                ocr_result["rec_scores"],
                ocr_result["text_word"],
                [wb.tolist() for wb in ocr_result["text_word_boxes"]]
            )),
            "preprocessed_image": page_image,
            "preprocessed_path": preprocessed_path,
            "timing": {
                "preprocess_seconds": round(t1 - t0, 3),
                "ocr_seconds": round(t2 - t1, 3),
                "total_seconds": round(t2 - t0, 3),
            },
        }
        page_data["markdown_path"] = save_to_markdown(page_data, page_path, page_dir)
        page_data["json_path"] = save_to_json(page_data, page_path, page_dir)
        pages.append(page_data)

    return pages

## Benchmark

In [87]:
pages = ingest_document(INPUT_DOC)

In [88]:
for i, page in enumerate(pages):
    print(f"page {i}")
    print(f"text lines: {len(page['lines'])}")
    print(f"saved: {page['markdown_path']}")
    print(f"timing: {page['timing']}")

# benchmark summary: which stage is heaviest, averaged across pages
stages = ["preprocess_seconds", "ocr_seconds", "total_seconds"]
totals = {s: sum(p["timing"][s] for p in pages) for s in stages}
averages = {s: round(totals[s] / len(pages), 3) for s in stages}

print(f"{'stage':<20}{'total (s)':<12}{'avg/page (s)':<14}{'% of total':<10}")
for s in ["preprocess_seconds", "ocr_seconds"]:
    pct = 100 * totals[s] / totals["total_seconds"]
    print(f"{s:<20}{round(totals[s], 3):<12}{averages[s]:<14}{pct:.1f}%")
print(f"{'total':<20}{round(totals['total_seconds'], 3):<12}{averages['total_seconds']:<14}")

page 0
text lines: 66
saved: outputs/invoice-sample-3/page-0/page-0.md
timing: {'preprocess_seconds': 0.62, 'ocr_seconds': 8.465, 'total_seconds': 9.085}
stage               total (s)   avg/page (s)  % of total
preprocess_seconds  0.62        0.62          6.8%
ocr_seconds         8.465       8.465         93.2%
total               9.085       9.085         
